# 🚗 Car Prediction using Ml algos
**Dataset:** Take default dataset of sklearn

**Goal:** Predict the price of a used car based on the selected features


---


## Step 1 — Import Libraries

**Why?**  
Before doing anything, we load the tools we need.
- `pandas` → to load and work with data (like Excel in Python)
- `numpy` → for math operations
- `matplotlib` & `seaborn` → to make graphs
- `sklearn` → machine learning models

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error,r2_score
from sklearn.metrics import accuracy_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline


## Step 2 — Load the Dataset

fetch_openml → Used to load datasets from OpenML.
name="autos" → Specifies the Automobile dataset.
version=1 → Loads version 1 of the dataset.
as_frame=True → Loads the dataset as a Pandas DataFrame.
automobile.frame → Accesses the DataFrame containing the dataset.
df → Stores the dataset for further analysis.
df.head() → Displays the first 5 rows of the dataset.
print(df) → Displays the complete dataset.

In [ ]:
from sklearn.datasets import fetch_openml
automobile=fetch_openml(name="autos",version=1,as_frame=True,parser="auto")
df=automobile.frame
print("dataset loaded")
# print(df.head())
print(df)

dataset loaded
     normalized-losses         make fuel-type aspiration num-of-doors  \
0                  NaN  alfa-romero       gas        std          two   
1                  NaN  alfa-romero       gas        std          two   
2                  NaN  alfa-romero       gas        std          two   
3                164.0         audi       gas        std         four   
4                164.0         audi       gas        std         four   
..                 ...          ...       ...        ...          ...   
200               95.0        volvo       gas        std         four   
201               95.0        volvo       gas      turbo         four   
202               95.0        volvo       gas        std         four   
203               95.0        volvo    diesel      turbo         four   
204               95.0        volvo       gas      turbo         four   

      body-style drive-wheels engine-location  wheel-base  length  ...  \
0    convertible          rwd     

##Step 3 — Convert Data Types

Why?
We convert the price, engine-size, and horsepower columns into numeric format so they can be used for calculations and analysis.

pd.to_numeric() converts the values into numbers.
errors="coerce" converts invalid values into NaN.
dtype is used to check the data type of a column.

⚠️ Correct data types are important for accurate data analysis.

In [ ]:
(df['price'])=pd.to_numeric(df['price'],errors="coerce")
df['price'].dtype
df['engine-size']=pd.to_numeric(df['engine-size'],errors='coerce')
df['horsepower']=pd.to_numeric(df['horsepower'],errors='coerce')




##Step 4 — Handle Missing Values & Split Data

Why?
We handle missing values by filling normalized-losses with its mean and removing rows where price is missing.

Then we split the data into:

X → Independent/Input variables
Y → Dependent/Target variable (price)

⚠️ price is removed from X because it is the value we want to predict.

In [ ]:
print(df.isnull().sum().sum())
df['normalized-losses']=df['normalized-losses'].fillna(df['normalized-losses'].mean())
df['normalized-losses']
df=df.dropna(subset=['price'])


# Dataset ko divide kro depepndent or indepepndent variables mai x= input y=o/p
X=df.drop(columns=['price'] )
# isme x m se price hatayenge kuki
# agar remove ni krnege toh voh bhi independent variables ban
# jayega and fhir model ko answer phele hi mil jyga

Y=df['price'].astype(float)

59


##Step 5 — Select Features

Why?
We select the important features from the dataset for model training.

Numerical features → engine-size, horsepower, curb-weight, etc.
Categorical features → make, fuel-type, body-style, and drive-wheels.

Finally, we combine both types of features in X.

⚠️ Selecting relevant features helps the model focus on useful information.

In [ ]:
# numerical features
num_features=[
    'engine-size',
    'horsepower',
    'curb-weight',
    'highway-mpg',
    'width',
    'length']

# categorical features
cat_features=['make',
              'fuel-type',
              'body-style',
              'drive-wheels' ]
X=X[num_features + cat_features]

##Step 6 — Preprocess Features

Why?
We prepare numerical and categorical data before giving it to the model.

Numerical columns → Missing values are filled using the median.
Categorical columns → Missing values are filled using the most frequent value.
OneHotEncoder → Converts categorical values into separate numerical columns.
Pipeline → Combines these preprocessing steps into the model workflow.

⚠️ Preprocessing makes the data suitable for machine learning.

In [ ]:
num_transformer=Pipeline(steps=[('imputer',SimpleImputer(strategy='median'))])
# isme agar numerical column mai column miss hoga toh median se fill hoga
# why pipeline is used ?
# jisse preprocessing ka step model ke workflow ka part ban jaye
cat_transformer=Pipeline(steps=[
    ('imputer',SimpleImputer(strategy='most_frequent')),
  # kaam iska yeh h ki agar column m nan a
    # jye toh most frequent value se fill kr denge
    ('encoder',OneHotEncoder(handle_unknown='ignore',
                             sparse_output=False))])
# onehot ka kaam h ki column wise divide kr dega kitne suvs h kitne sedean aise


##Step 7 — Apply Preprocessing

Why?
ColumnTransformer tells the model which preprocessing to apply to which columns.

Numerical columns → Median imputation
Categorical columns → Most frequent imputation + One-Hot Encoding

⚠️ It combines different preprocessing steps for numerical and categorical data.

In [ ]:
preprocessor=ColumnTransformer(transformers=[
    ('num',num_transformer,num_features),
    ("cat",cat_transformer, cat_features)
])
# iska kaam h btana ki konse column pr konsa preprocessing lagana hai
# ex -->( numerical --- median) and (categorical -- most frequent --> onehot)

##Step 8 — Create Model Pipeline

Why?
We combine the preprocessing and Linear Regression model into one pipeline.

preprocessor → Cleans and transforms the data.
LinearRegression() → Predicts the car price.

⚠️ The pipeline ensures preprocessing is done automatically before prediction.

In [ ]:
model_pipeline=Pipeline(steps=[('preprocessor',preprocessor),
                               ('regressor',LinearRegression())])

## Step 9 — Split the Dataset

**Why?**
We divide the dataset into **training and testing data**.

* **70% data** → Training the model
* **30% data** → Testing the model
* `random_state=39` → Keeps the split consistent.

> ⚠️ Training data is used to learn, while testing data is used to evaluate the model.


In [ ]:
X_train , X_test ,Y_train, Y_test=train_test_split(
    X,Y,test_size=0.3,random_state=39 # test_size 0.3 means 30% test 70% train
)


## Step 10 — Train the Model

**Why?**
We train the Linear Regression model using the training data.

* `X_train` → Input features
* `Y_train` → Actual car prices
* `.fit()` → Trains the model and learns the relationship between features and price.

> ⚠️ The trained model can now be used to predict car prices.


In [ ]:
model_pipeline.fit(X_train,Y_train)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['engine-size', 'horsepower',
                                                   'curb-weight', 'highway-mpg',
                                                   'width', 'length']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['make', 'fuel-type',
                                                   'body-style',
                                                   'drive-wheels'])])),
                ('regressor', LinearRegression())])

## Step 11 — Make Predictions

**Why?**
We use the trained model to predict car prices for the test data.

* `X_test` → Test input data
* `Y_pred` → Predicted car prices

> ⚠️ These predictions will be compared with the actual prices (`Y_test`) to evaluate the model.


In [ ]:
Y_pred=model_pipeline.predict(X_test)
Y_pred

array([32676.90354178,  5419.79638568, 16974.0765954 , 11379.48021682,
       17368.45895328,  5856.25201781, 16406.09517887,  9190.14538502,
       14351.71290289, 18270.07353213,  5229.30620367, 16073.35569438,
       15809.67549032, 20169.21580269, 16539.93221315,  7351.03004721,
        8476.79423987, 10520.24895862, 15807.09252829,  6773.33728448,
        6161.96544933, 12107.71338349, 10010.00333586,  5210.57142941,
        7513.15558033,  6283.82174254, 15249.67677191,  7030.08759964,
       14732.10379102,  6841.26702267,  9918.31473968, 29593.43901335,
        6740.99667482, 29772.03315331, 12251.62556272, 15692.53135672,
       20797.53925873,  8620.20896331,  6606.11880078, 17412.67272945,
       10238.50663306,  1363.68675283, 10888.63982899,  7661.95240286,
        5116.23563986, 19435.88556672, 17125.71690445,  9656.32359515,
       15362.33636375, 11337.23943624, 15051.68916378,  9146.73793075,
       12157.23268997,  9174.1732542 ,  5618.82931804,  6095.28936437,
      

## Step 12 — Evaluate the Model

**Why?**
We use the **R² Score** to measure how well our model predicts car prices.

* `Y_test` → Actual prices
* `Y_pred` → Predicted prices
* `r2_score()` → Calculates the model's R² score.

> ⚠️ A higher R² score means the model explains more of the variation in the target price.


In [ ]:
r2=r2_score(Y_test,Y_pred)
r2

0.8806377551929487

step 13 : MEAN ABSOLUTE ERROR


In [ ]:
mae=mean_absolute_error(Y_test,Y_pred)
mae

1560.722219531792

Step14 :  print the result

In [ ]:
print(f"R2 score (Coefficient of determination) : {r2:.2f}")
print(f'Mean Absolute Error : ${mae:.2f}')
# .2f means upto two decimal place

R2 score (Coefficient of determination) : 0.88
Mean Absolute Error : $1560.72


FINAL STEP  : Take your input data

In [ ]:
sample_car = pd.DataFrame([{
    'engine-size': 130,
    'horsepower': 111,
    'curb-weight': 2500,
    'highway-mpg': 27,
    'width': 65.5,
    'length': 168.8,
    'make': 'toyota',
    'fuel-type': 'gas',
    'body-style': 'sedan',
    'drive-wheels': 'fwd'
}])
predicted_price=model_pipeline.predict(sample_car)

In [ ]:
print(f"predicted price for the sample car: ${predicted_price[0]:.2f}")

predicted price for the sample car: $11340.93
